# 📊 E-Commerce Sales Analytics using Pandas
## Importing Libraries & Loading Dataset
Imports the Pandas library and loads the dataset into a DataFrame.  
It also converts the `order_date` column into datetime format for time-series analysis.

- `import pandas as pd` → Imports Pandas library
- `pd.read_csv()` → Reads CSV file
- `parse_dates=[]` → Converts column into datetime format
- `df.info()` → Shows dataset structure and data types

In [1]:
# Import pandas library
import pandas as pd

In [2]:
# Load dataset and convert order_date into datetime format
df = pd.read_csv(
    "ecommerce_sales_analytics_5000.csv",
    parse_dates=["order_date"]
)

In [3]:
# Display dataset information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          5000 non-null   int64         
 1   order_date        5000 non-null   datetime64[ns]
 2   customer_id       5000 non-null   int64         
 3   product_category  5000 non-null   object        
 4   region            5000 non-null   object        
 5   quantity          5000 non-null   int64         
 6   unit_price        5000 non-null   float64       
 7   discount          5000 non-null   float64       
 8   payment_method    5000 non-null   object        
 9   delivery_days     5000 non-null   int64         
 10  customer_rating   5000 non-null   float64       
 11  revenue           5000 non-null   float64       
dtypes: datetime64[ns](1), float64(4), int64(4), object(3)
memory usage: 468.9+ KB


# 💳 Creating Payment Type Categories

This section groups payment methods into broader categories:
- Card & Wallet → Digital
- COD → Cash

This helps simplify analysis.

- `map()` → Maps old values to new categories
- `unique()` → Shows unique values
- `groupby()` → Groups data
- `size()` → Counts rows in each group

In [4]:
# Create new payment type column
df["payment_type"] = df["payment_method"].map({
    "Card": "Digital",
    "Wallet": "Digital",
    "COD": "Cash"
})

In [5]:
# Show unique payment types
df["payment_type"].unique()

array(['Digital', 'Cash'], dtype=object)

In [6]:
# Preview payment columns
df[["payment_method", "payment_type"]].head()

,payment_method,payment_type
0,Wallet,Digital
1,Card,Digital
2,COD,Cash
3,Wallet,Digital
4,Wallet,Digital


In [7]:
# Count records for each payment type
df.groupby("payment_type").size()

payment_type
Cash       1774
Digital    3226
dtype: int64

# 💰 Revenue Category Classification

This section categorizes revenue into:
- High Revenue: Revenue > 500 → High
- Medium Revenue: Revenue between 200 and 500 → Medium
- Low Revenue: Revenue < 200 → Low

### Commands Used
- `apply()` → Applies function row-wise
- `value_counts()` → Counts category frequency
- `sample()` → Shows random rows

In [8]:
# Function to classify revenue
def revenue_category(row):
    # High revenue
    if row["revenue"] > 500:
        return "High"
    # Medium revenue
    elif row["revenue"] >= 200:
        return "Medium"
    # Low revenue
    else:
        return "Low"

In [9]:
# Apply function on dataset
df["revenue_category"] = df.apply(
    revenue_category,
    axis=1
)

In [10]:
# Count revenue categories
df["revenue_category"].value_counts()

revenue_category
High      3252
Medium    1098
Low        650
Name: count, dtype: int64

In [11]:
# Display sample records
df[["revenue", "revenue_category"]].sample(10)

,revenue,revenue_category
173,84.26,Low
2807,548.90,High
102,3449.50,High
4393,909.48,High
1446,392.24,Medium
1579,639.16,High
4701,3078.50,High
275,264.59,Medium
4693,34.05,Low
101,1132.34,High


# 📈 Creating Profit & Net Revenue using Pipe

This section creates:
- Profit column
- Net Revenue column

Using Pandas `pipe()` for clean workflow chaining.

## Formula Used
- Profit = Revenue × 20%
- Net Revenue = Revenue − Profit

### Commands Used
- `pipe()` → Passes DataFrame into functions
- `return` → Returns modified DataFrame

In [12]:
# Function to calculate profit
def add_profit(data):
    # Profit = 20% of revenue
    data["profit"] = data["revenue"] * 0.20
    return data

In [13]:
# Function to calculate net revenue
def add_net_revenue(data):
    # Net revenue after profit deduction
    data["net_revenue"] = (
        data["revenue"] - data["profit"]
    )
    return data

In [14]:
# Apply functions using pipe
df = (
    df
    .pipe(add_profit)
    .pipe(add_net_revenue)
)

In [15]:
# Preview calculated columns
df[["revenue", "profit", "net_revenue"]].head()

,revenue,profit,net_revenue
0,1883.20,376.640,1506.560
1,304.10,60.820,243.280
2,644.35,128.870,515.480
3,2569.90,513.980,2055.920
4,468.56,93.712,374.848


# 📊 Dataset Shape & Statistical Summary

This section explores:
- Dataset dimensions
- Statistical summary
- Revenue distribution

### Commands Used
- `shape` → Returns rows & columns
- `describe()` → Generates statistics
- `quantile()` → Calculates quartiles
- `mean()` → Average value
- `median()` → Middle value
- `skew()` → Checks data skewness

In [16]:
# Dataset dimensions
df.shape

(5000, 16)

In [17]:
# Statistical summary
df.describe()

,order_id,order_date,customer_id,quantity,unit_price,discount,delivery_days,customer_rating,revenue,profit,net_revenue
count,5000.000000,5000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,12500.500000,2028-11-04 12:00:00,1505.701200,4.044800,308.418774,0.179984,6.118800,2.973980,1021.955148,204.391030,817.564118
min,10001.000000,2022-01-01 00:00:00,1000.000000,1.000000,15.150000,0.000000,1.000000,1.000000,11.210000,2.242000,8.968000
25%,11250.750000,2025-06-03 18:00:00,1253.000000,2.000000,161.895000,0.090000,3.000000,2.000000,354.527500,70.905500,283.622000
50%,12500.500000,2028-11-04 12:00:00,1510.000000,4.000000,309.890000,0.180000,6.000000,3.000000,796.650000,159.330000,637.320000
75%,13750.250000,2032-04-07 06:00:00,1761.000000,6.000000,455.557500,0.270000,9.000000,4.000000,1515.690000,303.138000,1212.552000
max,15000.000000,2035-09-09 00:00:00,1999.000000,7.000000,599.960000,0.350000,11.000000,5.000000,4119.330000,823.866000,3295.464000
std,1443.520003,NaN,290.836902,2.020398,169.259369,0.101404,3.153264,1.157722,825.584219,165.116844,660.467375


In [18]:
# Summary for selected columns
df[["unit_price", "revenue"]].describe()

,unit_price,revenue
count,5000.000000,5000.000000
mean,308.418774,1021.955148
std,169.259369,825.584219
min,15.150000,11.210000
25%,161.895000,354.527500
50%,309.890000,796.650000
75%,455.557500,1515.690000
max,599.960000,4119.330000


In [19]:
# Revenue quartiles
df["revenue"].quantile([0.25, 0.5, 0.75])

0.25     354.5275
0.50     796.6500
0.75    1515.6900
Name: revenue, dtype: float64

In [20]:
# Mean and median
df["revenue"].mean(), df["revenue"].median()

(1021.955148, 796.65)

In [21]:
# Revenue skewness
df["revenue"].skew()

1.0017064603000962

In [22]:
# Average revenue by category
df.groupby("product_category")["revenue"].mean()

product_category
Beauty         1059.281992
Clothing       1000.608570
Electronics    1029.768835
Home           1013.502497
Name: revenue, dtype: float64

# 🚨 Outlier Detection using IQR Method
Outliers are unusual or extreme values in a dataset that are very different from other observations.

These values may occur because of:
- Data entry mistakes
- Measurement errors
- Rare events
- Natural variation in data

### 1. IQR Method (Interquartile Range)
- Skewed data
- Non-normal distributions
- IQR = Q3 − Q1

### Outlier Rule

- Value < Lower Bound → Outlier
- Value > Upper Bound → Outlier

### Commands Used
- `quantile()` → Finds quartiles
- Boolean filtering → Filters outliers
- `shape` → Counts rows

In [23]:
# Calculate quartiles
Q1 = df["revenue"].quantile(0.25)
Q3 = df["revenue"].quantile(0.75)

# Calculate IQR
IQR = Q3 - Q1

In [24]:
# Display values
Q1, Q3, IQR

(354.5275, 1515.69, 1161.1625000000001)

In [25]:
# Check calculations
(Q3 > Q1) and (IQR > 0)

True

In [26]:
# Define lower and upper bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

In [27]:
# Detect outliers
outliers_iqr = df[
    (df["revenue"] < lower_bound) |
    (df["revenue"] > upper_bound)
]

In [28]:
# Number of outliers
outliers_iqr.shape

(67, 16)

In [29]:
# Preview outliers
outliers_iqr[
    ["revenue", "product_category"]
].head()

,revenue,product_category
39,3640.56,Home
69,3338.37,Beauty
102,3449.50,Clothing
126,3472.39,Clothing
157,3917.54,Electronics


In [30]:
# Remove outliers
df = df[
    (df["revenue"] >= lower_bound) &
    (df["revenue"] <= upper_bound)
]

In [31]:
# Dataset shape after cleaning
df.shape

(4933, 16)

### 2. Z-Score Method

This section detects outliers using the Z-Score Method.
- Normally distributed data
- Z-score measures how many standard deviations a value is away from the mean.
- `|Z| > 3` → Outlier

### Commands Used
- `zscore()` → Calculates z-score
- `abs()` → Absolute value
- Boolean filtering → Removes outliers

In [32]:
# Import zscore function
from scipy.stats import zscore

In [34]:
# Reload dataset
df = pd.read_csv(
    "ecommerce_sales_analytics_5000-2.csv",
    parse_dates=["order_date"]
)

In [37]:
# Calculate z-score
df["revenue_z"] = zscore(df["revenue"])

In [38]:
# Detect outliers
outlier_z = df[
    df["revenue_z"].abs() > 3
]

In [39]:
# Number of outliers
outlier_z.shape

(26, 13)

In [40]:
# Remove outliers
df = df[
    df["revenue_z"].abs() <= 3
]

In [41]:
# Dataset shape after cleaning
df.shape

(4974, 13)

# 📅 Time Series Analysis using Resampling

This section performs monthly revenue analysis using datetime indexing and resampling.
- `set_index()` → Sets datetime column as index
- `resample("ME")` → Monthly-end resampling
- `sum()` → Aggregates revenue
- `pct_change()` → Percentage change

In [42]:
# Set order_date as index
df = df.set_index("order_date")

In [43]:
# Display index
df.index

DatetimeIndex(['2022-01-01', '2022-01-02', '2022-01-03', '2022-01-04',
               '2022-01-05', '2022-01-06', '2022-01-07', '2022-01-08',
               '2022-01-09', '2022-01-10',
               ...
               '2035-08-31', '2035-09-01', '2035-09-02', '2035-09-03',
               '2035-09-04', '2035-09-05', '2035-09-06', '2035-09-07',
               '2035-09-08', '2035-09-09'],
              dtype='datetime64[ns]', name='order_date', length=4974, freq=None)

In [44]:
# Monthly revenue analysis
monthly_revenue = (
    df["revenue"]
    .resample("ME")
    .sum()
)

In [45]:
# Display monthly revenue
monthly_revenue

order_date
2022-01-31    35624.89
2022-02-28    26268.33
2022-03-31    30309.28
2022-04-30    30781.96
2022-05-31    41629.24
                ...   
2035-05-31    29310.70
2035-06-30    25205.26
2035-07-31    23944.48
2035-08-31    29354.74
2035-09-30     4952.23
Freq: ME, Name: revenue, Length: 165, dtype: float64

In [46]:
# Monthly growth rate
monthly_revenue.pct_change()

order_date
2022-01-31         NaN
2022-02-28   -0.262641
2022-03-31    0.153834
2022-04-30    0.015595
2022-05-31    0.352391
                ...   
2035-05-31    0.054788
2035-06-30   -0.140066
2035-07-31   -0.050021
2035-08-31    0.225950
2035-09-30   -0.831297
Freq: ME, Name: revenue, Length: 165, dtype: float64

In [47]:
# Check sorted index
monthly_revenue.index.is_monotonic_increasing

True

In [48]:
# Compare totals
print(
    monthly_revenue.sum(),
    df["revenue"].sum()
)

5011909.1899999995 5011909.19


In [49]:
# Validation check
monthly_revenue.sum() <= df["revenue"].sum()

True

# 🔗 Covariance & Correlation Analysis

This section analyzes relationships between numerical variables using:
- Covariance
- Correlation
  
## Covariance 
- measures the relationship between two variables.
- Positive Covariance: Whether variables move together (x ↑  →  y ↑)
- Negative Covariance: Whether they move in opposite directions (x ↑  →  y ↓)
- Zero Covariance: No relationship between variables (x ↔ y)

## Correlation
It is the standardized version of covariance. Correlation measures Strength and Direction of relationship
- `+1` → Strong positive relation
- `-1` → Strong negative relation
- `0` → No relation

### Commands Used
- `cov()` → Covariance matrix
- `corr()` → Correlation matrix
- `sort_values()` → Sort correlations

In [50]:
# Select numerical columns
numeric_df = df[
    [
        "quantity",
        "unit_price",
        "discount",
        "delivery_days",
        "quantity",
        "revenue"
    ]
]

In [51]:
# Covariance matrix
numeric_df.cov()

,quantity,unit_price,discount,delivery_days,quantity,revenue
quantity,4.057453,-3.554419,0.001031,-0.054180,4.057453,1002.959663
unit_price,-3.554419,28434.609259,0.395646,8.522237,-3.554419,91454.701455
discount,0.001031,0.395646,0.010247,-0.000194,0.001031,-9.922703
delivery_days,-0.054180,8.522237,-0.000194,9.932649,-0.054180,15.784138
quantity,4.057453,-3.554419,0.001031,-0.054180,4.057453,1002.959663
revenue,1002.959663,91454.701455,-9.922703,15.784138,1002.959663,645471.483818


In [52]:
# Correlation matrix
numeric_df.corr()

,quantity,unit_price,discount,delivery_days,quantity,revenue
quantity,1.000000,-0.010464,0.005057,-0.008534,1.000000,0.619752
unit_price,-0.010464,1.000000,0.023178,0.016036,-0.010464,0.675062
discount,0.005057,0.023178,1.000000,-0.000607,0.005057,-0.122006
delivery_days,-0.008534,0.016036,-0.000607,1.000000,-0.008534,0.006234
quantity,1.000000,-0.010464,0.005057,-0.008534,1.000000,0.619752
revenue,0.619752,0.675062,-0.122006,0.006234,0.619752,1.000000


In [53]:
# Correlation with revenue
numeric_df.corr()["revenue"] \
    .sort_values(ascending=False)

revenue          1.000000
unit_price       0.675062
quantity         0.619752
quantity         0.619752
delivery_days    0.006234
discount        -0.122006
Name: revenue, dtype: float64

In [54]:
# Check correlation range
numeric_df.corr().values.min(), \
numeric_df.corr().values.max()

# Expected:
# Min >= -1
# Max <= +1

(-0.12200646754578048, 1.0)